# Step 5.2: Tweet tickers × trade tickers

The goal of this notebook is one dataset of Trump's social-media company references in 2025 and 2026 where every reference carries either a standardized ticker from the trade universe (`company.db`, Step 4) or an explicit "not in universe". The LLM (Step 5.1) gave us a company name and a ticker guess per reference; here we decide which ticker in the universe each one belongs to, the same way Step 2 decided tickers for the disclosures: deterministic rules first, a similarity match as a suggestion, and a manual review of everything the rules could not settle. Nothing in this notebook reads the trade dates. The logic flows as follows:

1. Data import and standardizing names and tickers
2. Rule-based mapping, in order; the first rule that fires sets `match_method`
3. Review of every pair in the universe
4. Final dataset
5. Descriptive analysis

In [1]:
import re
import sqlite3
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process

SOCIAL_DB = "../1) LabelTweets/SocialMedia.db"
COMPANY_DB = "../../4) CompanyDB/company.db"
review_path = "TickerTwitterReview.xlsx"
PRICES_CSV = "../../3) Prices/1) FetchPrices/prices.csv"   # daily prices of the universe (Step 3); used for the listing check in 3.2

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1) Data import and standardizing names and tickers

We import every company reference from Step 5.1 together with its post, and the ticker universe from Step 4. Both sides then get the same name and ticker standardization so that the join key means the same thing on both sides.

### 1.1) Data import

One row per post and company reference (`SocialMediaKeywords`), joined to the post it came from (`SocialMedia`) for the date, account and text. The universe is `companies` (one row per traded ticker with the two FMP names) plus `executives` (one row per ticker and executive), used only as a lookup for executive references without a usable company name.

In [2]:
with sqlite3.connect(SOCIAL_DB) as conn:
    refs = pd.read_sql_query(
        """SELECT k.rowid AS ref_id, k.ID, s.created_at, s.User, s.platform, s.text,
                  k.mention_text, k.entity_type, k.canonical_company, k.ticker_guess, k.reference_type,
                  k.sentiment, k.relevance, k.confidence, s.llm_model, s.llm_prompt_version
           FROM SocialMediaKeywords k JOIN SocialMedia s ON s.ID = k.ID
           ORDER BY s.created_at, k.ID, k.rowid""", conn)
    posts_total = pd.read_sql_query("SELECT llm_status, count(*) AS n FROM SocialMedia GROUP BY 1", conn)

with sqlite3.connect(COMPANY_DB) as conn:
    companies = pd.read_sql_query("SELECT ticker, fmp_company_name, profile_company_name, sector, industry FROM companies", conn)
    executives = pd.read_sql_query("SELECT ticker, name FROM executives", conn)

print(f"References loaded: {len(refs)} rows from {refs['ID'].nunique()} posts")
print(f"Universe loaded: {len(companies)} tickers, {len(executives)} executive rows")
print("\nPosts by LLM status:")
display(posts_total)
print("\nReferences by reference_type:")
display(refs["reference_type"].value_counts().rename_axis("reference_type").reset_index(name="rows"))

References loaded: 6468 rows from 4376 posts
Universe loaded: 1620 tickers, 16777 executive rows

Posts by LLM status:


,llm_status,n
0,content_filter,331
1,ok,21942



References by reference_type:


,reference_type,rows
0,indirect,3649
1,direct,2246
2,brand,412
3,executive,161


### 1.2) Standardizing company names

The Step 2.2 standardization ("Corporation" → "CORP", "Incorporated" → "INC", "Public Limited Company" → "PLC", upper case, punctuation stripped) is applied to the LLM's `canonical_company` and to the names of the universe: both FMP names and, since Step 2.5 moved renamed companies to their newest symbol, the SEC former names of each ticker's registrant from the Step 4.2 entity sources (so "IAC Inc." still meets `PPLI`, today People Inc., and "Madison Square Garden Sports Corp." meets `MSGK`). From the universe we build one lookup from standardized name to the set of tickers that carry it; a name that fits more than one ticker is the ambiguity case of rule 2.7 and is never resolved automatically.

In [3]:
def standardize_company(name) -> str:
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    s = str(name).upper().strip()
    s = s.replace(".", "").replace(",", "")
    s = re.sub(r"\bINCORPORATED\b", "INC", s)
    s = re.sub(r"\bCORPORATION\b", "CORP", s)
    s = re.sub(r"\bPUBLIC LIMITED COMPANY\b", "PLC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

refs["CompanyStd"] = refs["canonical_company"].apply(standardize_company)
companies["FmpStd"] = companies["fmp_company_name"].apply(standardize_company)
companies["ProfileStd"] = companies["profile_company_name"].apply(standardize_company)

# SEC former names of each universe ticker's registrant (Step 4.2 cache), standardized
import json
with sqlite3.connect(COMPANY_DB) as conn:
    cik_of_ticker = dict(conn.execute("SELECT ticker, entity_id FROM entities WHERE in_universe = 1 AND entity_id IS NOT NULL"))
    sec_payloads = dict(conn.execute("SELECT key, payload FROM entity_sources WHERE source = 'sec_submissions'"))
former_names = {}
for t, cik in cik_of_ticker.items():
    sub = json.loads(sec_payloads.get(cik) or "null") or {}
    # former names held in or after 2020 only: an older name (TEGNA was "Gannett Co., Inc." until 2015) belongs to another company today
    names = {standardize_company(sub.get("name"))} | {standardize_company(f.get("name")) for f in sub.get("formerNames", [])
                                                      if (f.get("to") or "")[:10] >= "2020-01-01"}
    former_names[t] = names - {""}
print(f"SEC names loaded for {len(former_names)} tickers ({sum(len(v) for v in former_names.values())} current and former names held since 2020)")

# standardized name -> set of tickers (over both FMP names and the SEC names)
name_to_tickers = {}
for _, r in companies.iterrows():
    for n in {r["FmpStd"], r["ProfileStd"]} | former_names.get(r["ticker"], set()):
        if n:
            name_to_tickers.setdefault(n, set()).add(r["ticker"])
ticker_to_names = {r["ticker"]: ({r["FmpStd"], r["ProfileStd"]} | former_names.get(r["ticker"], set())) - {""} for _, r in companies.iterrows()}
ticker_to_fmp = companies.set_index("ticker")["profile_company_name"].to_dict()

ambiguous_names = {n: t for n, t in name_to_tickers.items() if len(t) > 1}
print(f"{len(name_to_tickers)} distinct standardized universe names; {len(ambiguous_names)} fit more than one ticker:")
for n, t in sorted(ambiguous_names.items()):
    print(f"   {n}: {sorted(t)}")
display(refs[["canonical_company", "CompanyStd"]].drop_duplicates().head(8))

SEC names loaded for 1618 tickers (1910 current and former names held since 2020)
2177 distinct standardized universe names; 2 fit more than one ticker:
   FIRST BANCORP: ['FBNC', 'FBP']
   IAC/INTERACTIVECORP: ['MTCH', 'PPLI']


,canonical_company,CompanyStd
0,NaN,
2,ByteDance Ltd.,BYTEDANCE LTD
3,Oracle Corporation,ORACLE CORP
4,SoftBank Group Corp.,SOFTBANK GROUP CORP
5,"OpenAI, L.L.C.",OPENAI LLC
8,Stellantis N.V.,STELLANTIS NV
9,Fox Corporation,FOX CORP
10,Comcast Corporation,COMCAST CORP


### 1.3) Standardizing ticker guesses

The LLM writes the primary U.S. ticker as it knows it. The universe uses the FMP convention of Step 2.4 (one class per company, share classes with a hyphen), so the guess is upper-cased, stripped of `$`, dots and spaces become hyphens, and two small maps are applied, listed in full here:

* **share-class map**: the class the model prefers → the class in the universe (`GOOGL` → `GOOG`, `NWS` → `NWSA`, `BRK-A` / `BRK.A` → `BRK-B`);
* **rename map**: the same rule Step 2.5 applies to the trades, the latest symbol for every renamed ticker or merger successor. The chains come from `2) Trades/TickerReview.xlsx`, sheet `TickerRenames`, rows with `Decision = apply` (`PARA` → `PSKY` → `SKYD`, `BK` → `BNY`, `MMC` → `MRSH`, `INDV` → `SUPN`, ...), followed to their end, plus `EDR` → `TKO` (UFC moved from Endeavor to TKO Group; Endeavor was taken private in March 2025).

Both maps are extended from the data: after the mapping in 2) we list every guessed ticker that is not in the universe and check whether it is a class or rename of one that is. The original guess is kept in `ticker_guess`; the standardized one is `TickerGuessStd`.

In [4]:
SHARE_CLASS_MAP = {"GOOGL": "GOOG", "NWS": "NWSA", "BRK-A": "BRK-B", "BRKA": "BRK-B", "BF-A": "BF-B", "VOLV-B": "VOLV-B"}
# rename map: the Step 2.5 chains (latest symbol per renamed ticker or merger successor) plus the Endeavor -> TKO link
c8 = pd.read_excel("../../2) Trades/TickerReview.xlsx", sheet_name="TickerRenames")
RENAME_MAP = dict(zip(c8.loc[c8["Decision"] == "apply", "OldTicker"], c8.loc[c8["Decision"] == "apply", "LatestTicker"]))
RENAME_MAP.update({"PARA": "PSKY", "EDR": "TKO"})

def follow_renames(s):
    seen = set()
    while s in RENAME_MAP and s not in seen:
        seen.add(s); s = RENAME_MAP[s]
    return s

def standardize_ticker(t):
    if t is None or (isinstance(t, float) and np.isnan(t)) or str(t).strip() == "":
        return None
    s = str(t).upper().strip().lstrip("$")
    s = re.sub(r"[.\s/]+", "-", s)
    s = SHARE_CLASS_MAP.get(s, s)
    return follow_renames(s)

print(f"rename map: {len(RENAME_MAP)} entries from Step 2.5 (+ PARA, EDR):", ", ".join(f"{k}->{follow_renames(k)}" for k in sorted(RENAME_MAP)))

refs["TickerGuessStd"] = refs["ticker_guess"].apply(standardize_ticker)
universe = set(companies["ticker"])

changed = refs[refs["ticker_guess"].notna() & (refs["ticker_guess"].str.upper() != refs["TickerGuessStd"])]
print(f"Ticker guesses: {refs['ticker_guess'].notna().sum()} rows with a guess, {refs['TickerGuessStd'].isin(universe).sum()} of them in the universe after standardization")
print(f"Guesses changed by the standardization or the maps: {len(changed)} rows")
display(changed.groupby(["ticker_guess", "TickerGuessStd"]).size().reset_index(name="rows").sort_values("rows", ascending=False))

rename map: 23 entries from Step 2.5 (+ PARA, EDR): AMBC->OSG, ASGN->EFOR, ATGE->CVSA, AXL->DCH, BHLB->BBT, BK->BNY, CSWI->CSW, EDR->TKO, EQR->VMRK, HONAV->HONA, IAC->PPLI, MGPI->CASK, MMC->MRSH, MODG->CALY, MSGS->MSGK, NYMT->ADAM, OMI->ACH, PARA->SKYD, PSKY->SKYD, PSTG->P, SCVL->SHOE, SJW->HTO, VSNTV->VSNT
Ticker guesses: 1962 rows with a guess, 1693 of them in the universe after standardization
Guesses changed by the standardization or the maps: 173 rows


,ticker_guess,TickerGuessStd,rows
6,PARA,SKYD,81
1,EDR,TKO,51
2,GOOGL,GOOG,29
5,NWS,NWSA,6
4,MSGS,MSGK,3
0,BK,BNY,1
3,IAC,PPLI,1
7,VOLV B,VOLV-B,1


## 2) Rule-based mapping, in order; the first rule that fires sets `match_method`

Every reference goes through the rules of DataPlan Step 5.2.2 in the order a → g; the first one that fires sets `match_method` and, where allowed, `ticker`. Two of the rules are not stages of their own: indirect references (h) are set aside before the others, because they have no company to map; and the ambiguity rule (g) is applied inside the name match (b) and the similarity suggestion (e), which are the only places where a name can point at more than one ticker. Three methods assign a ticker automatically (`ticker+name`, `name`, `executive`); three only flag a row for review (`ticker_only`, `fuzzy`, `ambiguous`); the rest is `none`.

In [5]:
refs["ticker"] = None
refs["match_method"] = None
refs["CandidateTicker"] = None     # review rules: the ticker the rule points at
refs["CandidateCompany"] = None    # its FMP name
refs["Score"] = np.nan             # similarity score where one was computed
refs["Candidates"] = None          # ambiguous rows: every ticker the name fits

is_indirect = refs["reference_type"].eq("indirect")
open_rows = ~is_indirect           # rows no rule has fired on yet

def name_score(std_name, ticker):
    """Best token_set_ratio between the LLM's standardized name and the ticker's two standardized FMP names."""
    names = ticker_to_names.get(ticker) or set()
    return max((fuzz.token_set_ratio(std_name, n) for n in names), default=0)

def report(method):
    n = (refs["match_method"] == method).sum()
    print(f"{method}: {n} rows ({refs.loc[refs['match_method'] == method, 'ID'].nunique()} posts); still open: {open_rows.sum()} rows")

### 2.1) a) Ticker and name agree → `ticker+name`

The standardized guess exists in the universe and the LLM's standardized company name matches that ticker's FMP name exactly or with a `rapidfuzz` token-set score of at least 80. Two independent signals from the model agree with the universe, so the ticker is assigned without review.

In [6]:
NAME_AGREE_THRESHOLD = 80

in_universe = refs["TickerGuessStd"].isin(universe)
refs.loc[open_rows & in_universe, "Score"] = refs.loc[open_rows & in_universe].apply(
    lambda r: name_score(r["CompanyStd"], r["TickerGuessStd"]), axis=1)
fires = open_rows & in_universe & (refs["Score"] >= NAME_AGREE_THRESHOLD)
refs.loc[fires, "ticker"] = refs.loc[fires, "TickerGuessStd"]
refs.loc[fires, "match_method"] = "ticker+name"
open_rows &= ~fires
report("ticker+name")
display(refs.loc[fires, ["canonical_company", "TickerGuessStd", "Score"]].drop_duplicates().sort_values("Score").head(10))

ticker+name: 1611 rows (1385 posts); still open: 1208 rows


,canonical_company,TickerGuessStd,Score
1012,"Endeavor Group Holdings, Inc.",TKO,90.0
3,Oracle Corporation,ORCL,100.0
9,Fox Corporation,FOXA,100.0
8,Stellantis N.V.,STLA,100.0
11,"Warner Bros. Discovery, Inc.",WBD,100.0
21,General Motors Company,GM,100.0
22,"Tesla, Inc.",TSLA,100.0
10,Comcast Corporation,CMCSA,100.0
27,"Amazon.com, Inc.",AMZN,100.0
51,Lockheed Martin Corporation,LMT,100.0


### 2.2) b) Name match without a usable ticker → `name`, or `ambiguous`

The guess is empty or not in the universe, but the standardized company name is exactly one of the universe's standardized FMP names. If that name belongs to exactly one ticker, the ticker is assigned; if it fits more than one (rule g), the row is flagged `ambiguous` with every candidate and is never resolved automatically.

In [7]:
name_hits = refs["CompanyStd"].map(lambda n: name_to_tickers.get(n) if n else None)
has_name = open_rows & ~in_universe & name_hits.notna()

single = has_name & name_hits.map(lambda t: isinstance(t, set) and len(t) == 1)
refs.loc[single, "ticker"] = name_hits[single].map(lambda t: next(iter(t)))
refs.loc[single, "match_method"] = "name"

multi = has_name & name_hits.map(lambda t: isinstance(t, set) and len(t) > 1)
refs.loc[multi, "match_method"] = "ambiguous"
refs.loc[multi, "Candidates"] = name_hits[multi].map(lambda t: ", ".join(sorted(t)))
refs.loc[multi, "Score"] = 100

open_rows &= ~(single | multi)
report("name")
report("ambiguous")
display(refs.loc[single, ["canonical_company", "ticker_guess", "ticker"]].drop_duplicates().head(10))

name: 18 rows (18 posts); still open: 1190 rows
ambiguous: 0 rows (0 posts); still open: 1190 rows


,canonical_company,ticker_guess,ticker
236,"BGC Partners, Inc.",BGCP,BGC
439,Space Exploration Technologies Corp.,NaN,SPCX
1004,"Warrior Met Coal, Inc.",WMMC,HCC
3725,GlobalFoundries Inc.,NaN,GFS
5217,News Corporation,NEWS,NWSA
5422,Madison Square Garden Entertainment Corp.,MSGE,SPHR
5462,News Corp.,NEWS,NWSA


### 2.3) c) Executive match → `executive`

An `executive` reference whose company name did not match, but whose `mention_text` names a person in the universe's `executives` table for exactly one ticker. Names are compared on first and last name after dropping honorifics and handles, so that "Dave Ricks" meets "David A. Ricks" but a surname alone does not fire. One guard: when the LLM did name a company and that name has nothing in common with the matched ticker's FMP name (token-set score below 50), the person is a namesake, not that executive (Ken Griffin of Citadel is not the Ken Griffin on another company's board); such a row is sent to review as `fuzzy` with the ticker as candidate instead of being assigned.

In [8]:
HONORIFICS = {"MR", "MRS", "MS", "DR", "SIR", "CEO", "CHAIRMAN", "FOUNDER", "PRESIDENT"}

def person_key(name):
    """(first, last) of a personal name in upper case; None when fewer than two name tokens remain."""
    toks = [t for t in re.sub(r"[^A-Za-z\s]", " ", str(name)).upper().split() if t not in HONORIFICS and len(t) > 1]
    return (toks[0], toks[-1]) if len(toks) >= 2 else None

exec_index = {}
for _, r in executives.iterrows():
    k = person_key(r["name"])
    if k:
        exec_index.setdefault(k[1], []).append((k[0], r["ticker"]))

def executive_tickers(mention):
    """Tickers whose executive shares the mention's last name and a compatible first name."""
    toks = [t for t in re.sub(r"[^A-Za-z\s]", " ", str(mention)).upper().split() if t not in HONORIFICS and len(t) > 1]
    found = set()
    for i, last in enumerate(toks):
        for first, ticker in exec_index.get(last, []):
            if any(f == first or (len(f) >= 3 and len(first) >= 3 and (f.startswith(first[:3]) or first.startswith(f[:3]))) for f in toks[:i]):
                found.add(ticker)
    return found

EXEC_NAMESAKE_THRESHOLD = 50

is_exec = open_rows & refs["reference_type"].eq("executive")
exec_hits = refs.loc[is_exec, "mention_text"].apply(executive_tickers)
single_exec = exec_hits[exec_hits.map(len) == 1].map(lambda t: next(iter(t)))
namesake = single_exec.index[[refs.at[i, "CompanyStd"] != "" and name_score(refs.at[i, "CompanyStd"], t) < EXEC_NAMESAKE_THRESHOLD
                              for i, t in single_exec.items()]]
assigned = single_exec.drop(namesake)
refs.loc[assigned.index, "ticker"] = assigned
refs.loc[assigned.index, "match_method"] = "executive"
refs.loc[namesake, "match_method"] = "fuzzy"
refs.loc[namesake, "CandidateTicker"] = single_exec[namesake]
refs.loc[namesake, "CandidateCompany"] = single_exec[namesake].map(ticker_to_fmp)
refs.loc[namesake, "Score"] = [name_score(refs.at[i, "CompanyStd"], t) for i, t in single_exec[namesake].items()]
open_rows &= ~refs.index.isin(single_exec.index)
report("executive")
print(f"executive references checked: {is_exec.sum()}; matched one ticker: {len(single_exec)} "
      f"(assigned {len(assigned)}, sent to review as namesakes {len(namesake)}); matched several (left open): {(exec_hits.map(len) > 1).sum()}")
display(refs.loc[single_exec.index, ["mention_text", "canonical_company", "ticker_guess", "ticker", "match_method", "CandidateTicker", "CandidateCompany"]].drop_duplicates().head(10))

executive: 0 rows (0 posts); still open: 1189 rows
executive references checked: 20; matched one ticker: 1 (assigned 0, sent to review as namesakes 1); matched several (left open): 0


,mention_text,canonical_company,ticker_guess,ticker,match_method,CandidateTicker,CandidateCompany
3035,Ken Griffin,Citadel LLC,NaN,None,fuzzy,CDE,"Coeur Mining, Inc."


### 2.4) d) Ticker without name agreement → `ticker_only` (review)

The guess is in the universe but the names do not agree (score below 80). This is where the model puts the right company under a wrong ticker, or the right ticker under a name FMP spells very differently (Fox News → `FOXA` "Fox Corporation"). The guess becomes the candidate; the row is flagged and nothing is assigned.

In [9]:
fires = open_rows & in_universe
refs.loc[fires, "match_method"] = "ticker_only"
refs.loc[fires, "CandidateTicker"] = refs.loc[fires, "TickerGuessStd"]
refs.loc[fires, "CandidateCompany"] = refs.loc[fires, "TickerGuessStd"].map(ticker_to_fmp)
open_rows &= ~fires
report("ticker_only")
display(refs.loc[fires].groupby(["canonical_company", "CandidateTicker", "CandidateCompany"]).size().reset_index(name="rows").sort_values("rows", ascending=False).head(15))

ticker_only: 82 rows (81 posts); still open: 1107 rows


,canonical_company,CandidateTicker,CandidateCompany,rows
1,Paramount Global,SKYD,Skydance Corporation,76
2,"Paramount Global, Inc.",SKYD,Skydance Corporation,5
0,Nippon Steel Corporation,NSC,Norfolk Southern Corporation,1


### 2.5) e) Similarity suggestion → `fuzzy` (review), or `ambiguous`

For what is still open, the standardized company name is compared letter by letter (`rapidfuzz.fuzz.ratio`, as in Step 2.4.8) against every standardized universe name, both sides cut before the first legal-entity token (INC, CORP, PLC, CO, LTD, GROUP, HLDGS, HOLDINGS) so that "CoreWeave" meets "CoreWeave, Inc. Class A Common Stock". A best match of at least 85 is written as a suggestion with its score, never applied. Several tickers tying at the top score is the ambiguity case (rule g) and is flagged as such.

In [10]:
FUZZY_THRESHOLD = 85
BROAD_SUFFIX_TOKENS = ["INC", "CORP", "PLC", "CO", "LTD", "GROUP", "HLDGS", "HOLDINGS"]

def cut_before_any(name, tokens=BROAD_SUFFIX_TOKENS):
    """Cuts at whichever legal-entity token occurs first; falls back to the full name."""
    best_idx = None
    for token in tokens:
        m = re.search(rf"\b{token}\b", name)
        if m and (best_idx is None or m.start() < best_idx):
            best_idx = m.start()
    cut = name[:best_idx].strip().rstrip(",.").strip() if best_idx is not None else name
    return cut or name

prefix_to_tickers = {}
for n, t in name_to_tickers.items():
    prefix_to_tickers.setdefault(cut_before_any(n), set()).update(t)
universe_prefixes = list(prefix_to_tickers)

def best_matches(std_name):
    if not std_name:
        return []
    hits = process.extract(cut_before_any(std_name), universe_prefixes, scorer=fuzz.ratio, limit=5)
    hits = [(n, s) for n, s, _ in hits if s >= FUZZY_THRESHOLD]
    if not hits:
        return []
    top = hits[0][1]
    tickers = set().union(*(prefix_to_tickers[n] for n, s in hits if s == top))
    return [(n, s, sorted(tickers)) for n, s in hits if s == top]

cand = refs.loc[open_rows, "CompanyStd"].drop_duplicates()
suggestions = {n: best_matches(n) for n in cand}

for idx in refs.index[open_rows]:
    m = suggestions.get(refs.at[idx, "CompanyStd"]) or []
    if not m:
        continue
    tickers = m[0][2]
    refs.at[idx, "Score"] = m[0][1]
    if len(tickers) == 1:
        refs.at[idx, "match_method"] = "fuzzy"
        refs.at[idx, "CandidateTicker"] = tickers[0]
        refs.at[idx, "CandidateCompany"] = ticker_to_fmp.get(tickers[0])
    else:
        refs.at[idx, "match_method"] = "ambiguous"
        refs.at[idx, "Candidates"] = ", ".join(tickers)

fired = refs["match_method"].isin(["fuzzy", "ambiguous"]) & open_rows
open_rows &= ~fired
report("fuzzy")
report("ambiguous")
display(refs.loc[refs["match_method"] == "fuzzy"].groupby(["canonical_company", "CandidateTicker", "CandidateCompany", "Score"]).size().reset_index(name="rows").sort_values(["Score", "rows"], ascending=False).head(15))

fuzzy: 11 rows (11 posts); still open: 1097 rows
ambiguous: 0 rows (0 posts); still open: 1097 rows


,canonical_company,CandidateTicker,CandidateCompany,Score,rows
3,CoreWeave,CRWV,"CoreWeave, Inc. Class A Common Stock",100.000000,1
5,FTAI Aviation,FTAI,FTAI Aviation Ltd.,100.000000,1
8,Metallus,MTUS,Metallus Inc.,100.000000,1
4,Crest Management,ARES,Ares Management Corporation,90.322581,1
9,Penske Entertainment Corporation,PENN,"PENN Entertainment, Inc.",89.473684,1
1,"Beta Technologies, Inc.",UBER,"Uber Technologies, Inc.",88.235294,1
0,Atomic Industries,PATK,"Patrick Industries, Inc.",85.714286,1
6,Koch Industries,CF,"CF Industries Holdings, Inc.",85.714286,1
7,"Koch Industries, Inc.",CF,"CF Industries Holdings, Inc.",85.714286,1
10,Saronic,HWM,Howmet Aerospace Inc.,85.714286,1


### 2.6) f) Not in the universe → `none`

Nothing fired: the company is not one he traded (private firms, foreign media, NGOs, companies outside the 1,629). The row is kept with `ticker = NULL`; it is needed for the Part 1 share and the descriptive tables.

In [11]:
refs.loc[open_rows, "match_method"] = "none"
open_rows &= False
report("none")
unresolved = (refs[refs["match_method"] == "none"].groupby("canonical_company").size()
              .reset_index(name="rows").sort_values("rows", ascending=False))
print(f"{len(unresolved)} distinct company names without a match; the 30 most frequent:")
display(unresolved.head(30))

none: 1097 rows (945 posts); still open: 0 rows
414 distinct company names without a match; the 30 most frequent:


,canonical_company,rows
351,The New York Times Company,107
357,The Trump Organization,63
66,ByteDance Ltd.,40
359,The Washington Post,27
370,Trump Media & Technology Group Corp.,25
177,INDYCAR,21
407,X Corp.,15
237,"Newsmax Media, Inc.",15
178,IndyCar,15
368,Toyota Motor Corporation,14


### 2.7) h) Indirect references → `indirect`

Industry, commodity, group-of-companies and country statements have no company to map. They keep `mention_text` as the theme and `ticker = NULL`, and are carried along with `match_method = indirect` for the descriptive analysis and for the robustness check that expands industries to the universe's `sector` / `industry`.

In [12]:
refs.loc[is_indirect, "match_method"] = "indirect"
report("indirect")
assert refs["match_method"].notna().all(), "every row must have a match_method"
assert refs.loc[refs["ticker"].notna(), "ticker"].isin(universe).all(), "an assigned ticker is not in the universe"
assert refs.loc[refs["ticker"].notna(), "match_method"].isin(["ticker+name", "name", "executive"]).all()

indirect: 3649 rows (2395 posts); still open: 0 rows


### 2.8) Mapping summary

Rows and posts per `match_method` before the manual review, and the guessed tickers that are not in the universe, as a check on the two maps of 1.3.

In [13]:
summary = (refs.groupby("match_method").agg(rows=("ref_id", "size"), posts=("ID", "nunique"))
           .reindex(["ticker+name", "name", "executive", "ticker_only", "fuzzy", "ambiguous", "none", "indirect"]).fillna(0).astype(int))
summary["share_of_rows"] = (summary["rows"] / len(refs)).round(3)
display(summary)
print(f"Rows with a ticker assigned automatically: {refs['ticker'].notna().sum()} of {len(refs)}; flagged for review: {refs['match_method'].isin(['ticker_only', 'fuzzy', 'ambiguous']).sum()}")

not_in_universe = (refs[refs["TickerGuessStd"].notna() & ~refs["TickerGuessStd"].isin(universe)]
                   .groupby(["TickerGuessStd", "canonical_company"]).size().reset_index(name="rows").sort_values("rows", ascending=False))
print(f"\nGuessed tickers outside the universe: {not_in_universe['TickerGuessStd'].nunique()} distinct; the 25 most frequent (candidates for the share-class or rename map if they are a class or a rename of a universe ticker):")
display(not_in_universe.head(25))

,rows,posts,share_of_rows
match_method,,,
ticker+name,1611,1385,0.249
name,18,18,0.003
executive,0,0,0.000
ticker_only,82,81,0.013
fuzzy,11,11,0.002
ambiguous,0,0,0.000
none,1097,945,0.170
indirect,3649,2395,0.564


Rows with a ticker assigned automatically: 1629 of 6468; flagged for review: 93

Guessed tickers outside the universe: 47 distinct; the 25 most frequent (candidates for the share-class or rename map if they are a class or a rename of a universe ticker):


,TickerGuessStd,canonical_company,rows
26,NYT,The New York Times Company,107
7,DJT,Trump Media & Technology Group Corp.,25
35,TM,Toyota Motor Corporation,14
51,X,United States Steel Corporation,10
25,NXST,"Nexstar Media Group, Inc.",7
2,AZN,AstraZeneca PLC,6
8,DMGT,Daily Mail and General Trust plc,6
23,NVO,Novo Nordisk A/S,6
15,HMC,"Honda Motor Co., Ltd.",6
14,GCI,"Gannett Co., Inc.",6


## 3) Review of every pair in the universe

Nothing gets a universe ticker in the final dataset without a recorded review decision (DataPlan Step 5.2.3; `tasks/lessons.md`). This covers the pairs the rules accepted automatically (`ticker+name`, `name`, `executive`) as well as the flagged ones (`ticker_only`, `fuzzy`, `ambiguous`). Rule a's two signals both come from the LLM, so they can agree on the wrong company (Tylenol under Johnson & Johnson), and only a review catches that. The review works at two levels and is read from `TickerTwitterReview.xlsx`:

| Sheet | Written by | Content |
|---|---|---|
| `ReadMe` | this notebook, every run | how the workbook works |
| `AllPairs` | this notebook, every run | every distinct (`canonical_company`, `ticker_guess`) pair a rule gave or offered a universe ticker, plus every pair with a decision: rule, ticker, number of references, first and last post, all mentions, an example post, and its `Status` (decided / pending) |
| `ToReview` | this notebook, every run | the pending pairs of `AllPairs`; empty when the review is complete |
| `FlaggedRows` | this notebook, every run | every flagged reference with its full post text |
| `PairDecisions` | the reviewer, by hand | one decision per pair: `accept` (the candidate ticker), `correct` (another universe ticker, in `CorrectedTicker`) or `not_in_universe`, with `ReviewReason` and `Reviewer` |
| `RowOverrides` | the reviewer, by hand | exceptions for single references inside a pair, keyed on post `ID` and `mention_text`: `correct` or `not_in_universe` (Tylenol inside a Johnson & Johnson pair is Kenvue) |
| `Entities` | `4) CompanyDB/BuildEntities.py`, decisions by hand | identity events of the universe companies (Step 4.2) |

The code never edits the hand-written sheets. A rerun rewrites the generated sheets and applies the hand-written ones (3.2). After the decisions, a reference whose post date falls outside the price series of its ticker (`3) Prices/1) FetchPrices/prices.csv`) gets no ticker and `match_method = none_unlisted`, because the company was not listed that day: SpaceX before its listing on 2026-06-12, WK Kellogg after it was taken over (series ends 2025-09-25).

### 3.1) Writing the review workbook

In [14]:
assigning_methods = ["ticker+name", "name", "executive"]
flag_methods = ["ticker_only", "fuzzy", "ambiguous"]
PAIR_DECISIONS = {"accept", "correct", "not_in_universe"}
ROW_DECISIONS = {"correct", "not_in_universe"}
PAIR_COLS = ["canonical_company", "ticker_guess", "match_method", "CandidateTicker", "CandidateCompany", "Rows", "Posts",
             "ExampleMention", "ExampleID", "Decision", "CorrectedTicker", "ReviewReason", "Reviewer"]
ROW_COLS = ["ID", "created_at", "mention_text", "canonical_company", "ticker_guess", "RuleTicker",
            "Decision", "CorrectedTicker", "ReviewReason", "Reviewer"]

def read_hand_sheet(name, columns):
    """A hand-written sheet of the review workbook, all cells as text; an empty frame if the sheet does not exist yet."""
    try:
        df = pd.read_excel(review_path, sheet_name=name, dtype=str)
    except (FileNotFoundError, ValueError):
        return pd.DataFrame(columns=columns)
    missing = set(columns) - set(df.columns)
    assert not missing, f"sheet {name} lacks the columns {missing}"
    return df.dropna(how="all")

pair_decisions = read_hand_sheet("PairDecisions", PAIR_COLS)
row_overrides = read_hand_sheet("RowOverrides", ROW_COLS)
pair_decisions["key_company"] = pair_decisions["canonical_company"].fillna("")
pair_decisions["key_guess"] = pair_decisions["ticker_guess"].fillna("")
assert not pair_decisions.duplicated(["key_company", "key_guess"]).any(), "a pair has more than one row in PairDecisions"

refs["key_company"] = refs["canonical_company"].fillna("")
refs["key_guess"] = refs["ticker_guess"].fillna("").astype(str)
pair_key = pd.Series(list(zip(refs["key_company"], refs["key_guess"])), index=refs.index)
row_key = pd.Series(list(zip(refs["ID"].astype(str), refs["mention_text"].fillna(""))), index=refs.index)
decided_keys = set(zip(pair_decisions["key_company"], pair_decisions["key_guess"]))
override_keys = set(zip(row_overrides["ID"].astype(str), row_overrides["mention_text"].fillna("")))

# every pair a rule gave or offered a universe ticker, plus every pair the review decided (a decision can also overrule `none`)
in_pairs = ~is_indirect & (refs["match_method"].isin(assigning_methods + flag_methods) | pair_key.isin(decided_keys))
p = refs[in_pairs].copy()
p["RuleTicker"] = p["ticker"]
p["CandidateTicker"] = p["CandidateTicker"].fillna(p["ticker"])
p["CandidateCompany"] = p["CandidateCompany"].fillna(p["CandidateTicker"].map(ticker_to_fmp))
p["post_date"] = p["created_at"].str[:10]
p["has_override"] = row_key[in_pairs].isin(override_keys)

def mention_list(s, cap=700):
    return " | ".join(f"{m} ({n})" for m, n in s.fillna("").value_counts().items())[:cap]

all_pairs = (p.groupby(["key_company", "key_guess"])
             .agg(canonical_company=("canonical_company", "first"), ticker_guess=("ticker_guess", "first"),
                  match_method=("match_method", lambda s: ", ".join(sorted(set(s)))),
                  RuleTicker=("RuleTicker", "first"), CandidateTicker=("CandidateTicker", "first"),
                  CandidateCompany=("CandidateCompany", "first"), Candidates=("Candidates", "first"), Score=("Score", "max"),
                  Rows=("ref_id", "size"), Posts=("ID", "nunique"), RowOverrides=("has_override", "sum"),
                  FirstPost=("post_date", "min"), LastPost=("post_date", "max"), Mentions=("mention_text", mention_list),
                  ExampleMention=("mention_text", "first"), ExampleText=("text", "first"), ExampleID=("ID", "first"))
             .reset_index())
all_pairs["ExampleID"] = all_pairs["ExampleID"].astype(str)
all_pairs = all_pairs.merge(pair_decisions[["key_company", "key_guess", "Decision", "CorrectedTicker", "ReviewReason", "Reviewer"]],
                            on=["key_company", "key_guess"], how="left")
all_pairs["Status"] = np.where(all_pairs["Decision"].notna(), "decided", "pending")
all_pairs = (all_pairs.sort_values(["Status", "Rows"], ascending=[False, False])
             [["Status", "match_method", "canonical_company", "ticker_guess", "RuleTicker", "CandidateTicker", "CandidateCompany",
               "Candidates", "Score", "Rows", "Posts", "RowOverrides", "FirstPost", "LastPost", "Mentions", "ExampleMention",
               "ExampleText", "ExampleID", "Decision", "CorrectedTicker", "ReviewReason", "Reviewer"]])
to_review = all_pairs[all_pairs["Status"] == "pending"].drop(columns=["Decision", "CorrectedTicker", "ReviewReason", "Reviewer"])

flagged_rows = refs[refs["match_method"].isin(flag_methods)][
    ["ref_id", "ID", "created_at", "User", "platform", "text", "mention_text", "reference_type", "canonical_company",
     "ticker_guess", "TickerGuessStd", "match_method", "CandidateTicker", "CandidateCompany", "Candidates", "Score"]]

readme = pd.DataFrame([
    ("How it works", "Every company reference that a rule gave or offered a ticker from the trade universe belongs to a pair "
     "(canonical_company, ticker_guess). Nothing reaches Twitter_tickers_2025_2026.xlsx with a ticker unless its pair has a decision in PairDecisions. "
     "TwitterTickerCleaning.ipynb (section 3) applies the decisions on every run; nobody edits the data by hand."),
    ("Order of application", "1) the rules a-h give a ticker or a flag; 2) PairDecisions decides every pair; 3) RowOverrides changes single references "
     "inside a pair; 4) a reference dated outside the price series of its ticker (3) Prices/1) FetchPrices/prices.csv) loses the ticker: none_unlisted."),
    ("ReadMe, AllPairs, ToReview, FlaggedRows", "Generated by the notebook on every run; do not edit."),
    ("AllPairs", "Every pair with its rule, the ticker the rule gave (RuleTicker) or offered (CandidateTicker), number of references and posts, "
     "first and last post, all mentions with counts, one example post, how many references have a RowOverride, and Status: decided or pending."),
    ("ToReview", "The pending pairs of AllPairs. Empty when the review is complete; a new pair (after a change of the universe or the labels) appears here and gets no ticker until decided."),
    ("FlaggedRows", "Every reference a rule flagged (ticker_only, fuzzy, ambiguous) with its full post text."),
    ("PairDecisions (hand-written)", "One row per pair. Decision: accept = the CandidateTicker is right; correct = another universe ticker, written in "
     "CorrectedTicker; not_in_universe = no company of the trade universe. For accept and correct, CorrectedTicker holds the ticker. "
     "ReviewReason says why; Reviewer says who decided and when."),
    ("RowOverrides (hand-written)", "Exceptions for single references inside a pair, keyed on ID and mention_text (both must match the reference exactly). "
     "Decision: correct (CorrectedTicker) or not_in_universe. Example: Tylenol references in a Johnson & Johnson pair belong to Kenvue (KVUE)."),
    ("Entities (BuildEntities.py, decisions by hand)", "Identity events of the universe companies inside 2025-01-20 .. 2026-08-31 (Step 4.2). Decision: "
     "confirmed = the event and its date agree with the price series and the rename review; keep = flagged, but the same company; "
     "successor = the company continues under the successor's ticker; spun_off = new listing from a spin-off; "
     "unpriceable = no usable price series, purchases left out; noted = recorded, with what is open or why it has no effect, in ReviewNote."),
    ("Match methods in the final dataset", "ticker+name, name, executive = rule ticker, confirmed by the pair decision; manual = ticker set or corrected by the review; "
     "none_reviewed = the review found no universe company; none_unlisted = company not listed on the post date; none = no rule fired, not reviewed; "
     "indirect = industry or theme; ticker_only, fuzzy, ambiguous = flagged pair still pending (no ticker)."),
    ("Rule ordering", "Decisions are by pair, so one decision covers every reference with the same company name and ticker guess. "
     "A decision overrules whatever the rules did with the pair."),
], columns=["Item", "Explanation"])

import os
writer_kwargs = dict(mode="a", if_sheet_exists="replace") if os.path.exists(review_path) else dict(mode="w")
with pd.ExcelWriter(review_path, engine="openpyxl", **writer_kwargs) as writer:
    readme.to_excel(writer, sheet_name="ReadMe", index=False)
    all_pairs.to_excel(writer, sheet_name="AllPairs", index=False)
    to_review.to_excel(writer, sheet_name="ToReview", index=False)
    flagged_rows.to_excel(writer, sheet_name="FlaggedRows", index=False)
    if "PairDecisions" not in writer.book.sheetnames:
        pair_decisions[PAIR_COLS].to_excel(writer, sheet_name="PairDecisions", index=False)
    if "RowOverrides" not in writer.book.sheetnames:
        row_overrides[ROW_COLS].to_excel(writer, sheet_name="RowOverrides", index=False)

print(f"AllPairs: {len(all_pairs)} pairs covering {int(all_pairs['Rows'].sum())} references; "
      f"decided {int((all_pairs['Status'] == 'decided').sum())}, pending {len(to_review)} (ToReview); "
      f"{len(flagged_rows)} flagged references in FlaggedRows")
display(all_pairs.groupby(["Status", "match_method"]).agg(pairs=("canonical_company", "size"), references=("Rows", "sum")))
display(to_review.head(20))

AllPairs: 168 pairs covering 1723 references; decided 168, pending 0 (ToReview); 93 flagged references in FlaggedRows


pairs  references
Status  match_method                   
decided fuzzy            11          11
        name              7          18
        none              1           1
        ticker+name     146        1611
        ticker_only       3          82

,Status,match_method,canonical_company,ticker_guess,RuleTicker,CandidateTicker,CandidateCompany,Candidates,Score,Rows,Posts,RowOverrides,FirstPost,LastPost,Mentions,ExampleMention,ExampleText,ExampleID


### 3.2) Applying the decisions

`PairDecisions` is applied to every reference of a pair, whichever rule produced it: `accept` or `correct` → `ticker = CorrectedTicker` (a ticker renamed since the review follows the Step 2.5 chain, PSKY → SKYD); `not_in_universe` → no ticker, `match_method = none_reviewed`. An accepted automatic pair keeps the rule as its `match_method`; an accepted flagged pair, a correction, or a pair the rules had left unmatched becomes `manual`. `RowOverrides` then changes single references. Every decided reference gets `pair_reviewed = 1` and the reviewer's reason in `review_reason`. A pair without a decision gets no ticker and `pair_reviewed = 0`, and is listed in `ToReview` for the next review round. Last, a reference dated outside the price series of its ticker becomes `none_unlisted`. The notebook stops on an unknown decision, a corrected ticker outside the universe, an `accept` that names a ticker other than the candidate, or a `RowOverrides` key that matches no reference.

In [15]:
def norm_ticker(t):
    return follow_renames(str(t).strip().upper()) if isinstance(t, str) and t.strip() else None

dec = pair_decisions.copy()
dec["Decision"] = dec["Decision"].fillna("").str.strip().str.lower()
dec["CorrectedTicker"] = dec["CorrectedTicker"].map(norm_ticker)
dec["CandidateTicker"] = dec["CandidateTicker"].map(norm_ticker)
assert dec["Decision"].isin(PAIR_DECISIONS).all(), f"unknown Decision in PairDecisions: {set(dec['Decision']) - PAIR_DECISIONS}"
takes = dec["Decision"].isin(["accept", "correct"])
assert dec.loc[takes, "CorrectedTicker"].notna().all(), "accept/correct without a CorrectedTicker"
assert dec.loc[takes, "CorrectedTicker"].isin(universe).all(), f"CorrectedTicker not in the universe: {set(dec.loc[takes, 'CorrectedTicker']) - universe}"
bad_accept = dec[(dec["Decision"] == "accept") & (dec["CorrectedTicker"] != dec["CandidateTicker"])]
assert bad_accept.empty, f"an 'accept' names a ticker other than the candidate: {bad_accept[['canonical_company', 'CandidateTicker', 'CorrectedTicker']].values.tolist()}"
assert dec["ReviewReason"].notna().all() and dec["Reviewer"].notna().all(), "every decision needs a ReviewReason and a Reviewer"

ro = row_overrides.copy()
ro["ID"] = ro["ID"].astype(str)
ro["Decision"] = ro["Decision"].fillna("").str.strip().str.lower()
ro["CorrectedTicker"] = ro["CorrectedTicker"].map(norm_ticker)
assert ro["Decision"].isin(ROW_DECISIONS).all(), f"unknown Decision in RowOverrides: {set(ro['Decision']) - ROW_DECISIONS}"
assert ro.loc[ro["Decision"] == "correct", "CorrectedTicker"].isin(universe).all(), "a RowOverrides CorrectedTicker is not in the universe"
assert ro["ReviewReason"].notna().all() and ro["Reviewer"].notna().all(), "every override needs a ReviewReason and a Reviewer"
ro_keys = list(zip(ro["ID"], ro["mention_text"].fillna("")))
assert len(set(ro_keys)) == len(ro_keys), "a reference has more than one row in RowOverrides"
unmatched = set(ro_keys) - set(row_key)
assert not unmatched, f"RowOverrides rows that match no reference (ID, mention_text): {unmatched}"

rule_method = refs["match_method"].copy()
refs["pair_reviewed"] = 0
refs["review_reason"] = None

# pair decisions
dec_map = {(c, g): r for c, g, r in zip(dec["key_company"], dec["key_guess"], dec[["Decision", "CorrectedTicker", "ReviewReason"]].itertuples(index=False))}
decided = ~is_indirect & pair_key.isin(dec_map.keys())
d_dec = pair_key[decided].map(lambda k: dec_map[k].Decision)
d_tic = pair_key[decided].map(lambda k: dec_map[k].CorrectedTicker)
d_why = pair_key[decided].map(lambda k: dec_map[k].ReviewReason)
gets = d_dec.isin(["accept", "correct"])
refs.loc[gets[gets].index, "ticker"] = d_tic[gets]
keeps_rule = gets & d_dec.eq("accept") & rule_method[decided].isin(assigning_methods)
refs.loc[gets[gets & ~keeps_rule].index, "match_method"] = "manual"
refs.loc[gets[~gets].index, "ticker"] = None
refs.loc[gets[~gets].index, "match_method"] = "none_reviewed"
refs.loc[decided, "pair_reviewed"] = 1
refs.loc[decided, "review_reason"] = d_why

# pairs without a decision: no ticker until reviewed; the flag or the rule stays visible in match_method
pending_mask = ~decided & rule_method.isin(assigning_methods + flag_methods)
refs.loc[pending_mask, "ticker"] = None

# single references
ro_map = dict(zip(ro_keys, ro[["Decision", "CorrectedTicker", "ReviewReason"]].itertuples(index=False)))
overridden = row_key.isin(ro_map.keys())
for i in refs.index[overridden]:
    o = ro_map[row_key[i]]
    refs.at[i, "ticker"] = o.CorrectedTicker if o.Decision == "correct" else None
    refs.at[i, "match_method"] = "manual" if o.Decision == "correct" else "none_reviewed"
    refs.at[i, "pair_reviewed"] = 1
    refs.at[i, "review_reason"] = o.ReviewReason

# listing check: no universe ticker on a day outside its price series
prices = pd.read_csv(PRICES_CSV, usecols=["date", "ticker"])
series = prices.groupby("ticker")["date"].agg(first_price="min", last_price="max")
post_day = refs["created_at"].str[:10]
lo, hi = refs["ticker"].map(series["first_price"]), refs["ticker"].map(series["last_price"])
unlisted = refs["ticker"].notna() & lo.notna() & ((post_day < lo) | (post_day > hi))
refs.loc[unlisted, "review_reason"] = ("not listed on the post date: the " + refs.loc[unlisted, "ticker"] + " price series runs "
                                       + lo[unlisted] + " to " + hi[unlisted] + " (prices.csv)")
refs.loc[unlisted, "ticker"] = None
refs.loc[unlisted, "match_method"] = "none_unlisted"
no_series = sorted(set(refs.loc[refs["ticker"].notna() & ~refs["ticker"].isin(series.index), "ticker"]))

pending = (refs[pending_mask].groupby(["match_method", "canonical_company", "ticker_guess"], dropna=False)
           .size().reset_index(name="rows"))
print(f"PairDecisions: {len(dec)} pairs {dict(dec['Decision'].value_counts())}, applied to {int(decided.sum())} references")
print("  rule before -> method after:", refs.loc[decided].groupby([rule_method[decided], refs.loc[decided, 'match_method']]).size().to_dict())
print(f"RowOverrides: {len(ro)} references {dict(ro['Decision'].value_counts())}")
print(f"Not listed on the post date (none_unlisted): {int(unlisted.sum())} references", dict(refs.loc[unlisted, "review_reason"].str.extract(r"the (\S+) price")[0].value_counts()))
print(f"Universe tickers without a price series (kept, no listing check possible): {no_series}")
print(f"Pairs still without a decision: {len(pending)} pairs, {int(pending_mask.sum())} references (no ticker; see ToReview)")
display(pending)
display(refs.loc[overridden, ["created_at", "mention_text", "canonical_company", "ticker", "match_method", "review_reason"]])
refs = refs.drop(columns=["key_company", "key_guess"])

PairDecisions: 168 pairs {'accept': np.int64(151), 'not_in_universe': np.int64(16), 'correct': np.int64(1)}, applied to 1723 references
  rule before -> method after: {('fuzzy', 'manual'): 3, ('fuzzy', 'none_reviewed'): 8, ('name', 'name'): 7, ('name', 'none_reviewed'): 4, ('name', 'none_unlisted'): 7, ('none', 'manual'): 1, ('ticker+name', 'manual'): 19, ('ticker+name', 'none_reviewed'): 20, ('ticker+name', 'none_unlisted'): 1, ('ticker+name', 'ticker+name'): 1571, ('ticker_only', 'manual'): 81, ('ticker_only', 'none_reviewed'): 1}
RowOverrides: 30 references {'correct': np.int64(19), 'not_in_universe': np.int64(11)}
Not listed on the post date (none_unlisted): 8 references {'SPCX': np.int64(7), 'KLG': np.int64(1)}
Universe tickers without a price series (kept, no listing check possible): ['HOG']
Pairs still without a decision: 0 pairs, 0 references (no ticker; see ToReview)


,match_method,canonical_company,ticker_guess,rows


,created_at,mention_text,canonical_company,ticker,match_method,review_reason
671,2025-03-23T00:17:28Z,Wells Fargo Center,Wells Fargo & Company,None,none_reviewed,The Wells Fargo Center arena (NCAA wrestling championships); a naming-rights...
1652,2025-05-28T16:06:23Z,Coca-Cola 600,The Coca-Cola Company,None,none_reviewed,The Coca-Cola 600 is a NASCAR race; a sponsored event name is not a referenc...
1941,2025-06-21T00:04:57Z,Coca-Cola 600,The Coca-Cola Company,None,none_reviewed,The Coca-Cola 600 is a NASCAR race; a sponsored event name is not a referenc...
3151,2025-09-22T22:07:17Z,Tylenol,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
3154,2025-09-22T23:25:06Z,Tylenol,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
3203,2025-09-26T14:45:00Z,TYLENOL,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
3278,2025-10-08T04:10:00Z,TYLENOL,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
3425,2025-10-26T20:19:00Z,TYLENOL,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
4106,2026-01-05T22:19:00Z,TYLENOL,Johnson & Johnson,KVUE,manual,Tylenol is a Kenvue brand since the 2023 spin-off from Johnson & Johnson (po...
4282,2026-01-22T17:11:00Z,MSDNC,Comcast Corporation,VSNT,manual,MSNBC (renamed MS NOW) and CNBC left Comcast with the Versant Media spin-off...


## 4) Final dataset

One row per post and company reference, `Twitter_tickers_2025_2026.xlsx`. `ticker` is the universe ticker under its latest symbol, the key for the join with the trades (Step 2.5 renames, PSKY → SKYD). `ticker_at_post` is the symbol the company traded under on the post date, `entity_id` its SEC CIK (Step 4.2) and `company_final` its FMP name. `in_universe = 1` where a ticker is set. `pair_reviewed` and `review_reason` record the decision behind the row. The symbol history comes from the Step 2.5 renames with a date (`TickerReview.xlsx`, `TickerRenames`), the entity table's former tickers, and its merger successors (PARA → Paramount Skydance on 2025-08-07). Before the export we check that every reference from Step 5.1 is present; that no row has a ticker without a decision, a `match_method` that assigns one, a universe ticker and a price on its post date; and we print the counts per `match_method` that also go into the descriptive workbook.

In [16]:
with sqlite3.connect(COMPANY_DB) as conn:
    ent = pd.read_sql_query("SELECT entity_id, ticker, in_universe, valid_to, event_type, event_date, successor_entity FROM entities", conn)
universe_ticker_of_entity = {}
for t, cik in cik_of_ticker.items():
    universe_ticker_of_entity.setdefault(cik, t)

earlier_symbols = {}   # latest ticker -> [(earlier symbol, first day of the next symbol)]
def add_link(latest, old, until):
    if latest and old and isinstance(until, str) and until[:4].isdigit() and old != latest:
        if old not in {o for o, _ in earlier_symbols.get(latest, [])}:
            earlier_symbols.setdefault(latest, []).append((old, until[:10]))

for r in c8[(c8["Decision"] == "apply") & c8["ChangeDate"].notna()].itertuples():
    add_link(follow_renames(r.LatestTicker), r.OldTicker, str(r.ChangeDate))
for r in ent[(ent["in_universe"] == 0) & (ent["event_type"] == "renamed") & ent["valid_to"].notna()].itertuples():
    add_link(universe_ticker_of_entity.get(r.entity_id), r.ticker, r.valid_to)
for r in ent[(ent["event_type"] == "merged_into") & ent["successor_entity"].notna()].itertuples():
    add_link(universe_ticker_of_entity.get(r.successor_entity), r.ticker, r.event_date)

def ticker_at(latest, day):
    """Symbol of `latest`'s company on `day`: walk back through the symbol changes after that day."""
    if not isinstance(latest, str):
        return None
    current = latest
    for old, until in sorted(earlier_symbols.get(latest, []), key=lambda x: x[1], reverse=True):
        if day < until:
            current = old
    return current

print("Symbol histories used for ticker_at_post:", {k: sorted(v, key=lambda x: x[1]) for k, v in earlier_symbols.items() if k in set(refs["ticker"].dropna())})

refs["in_universe"] = refs["ticker"].notna().astype(int)
refs["ticker_at_post"] = [ticker_at(t, d) for t, d in zip(refs["ticker"], refs["created_at"].str[:10])]
refs["entity_id"] = refs["ticker"].map(cik_of_ticker)
refs["company_final"] = refs["ticker"].map(ticker_to_fmp)

final_cols = ["ID", "created_at", "User", "platform", "text", "mention_text", "canonical_company", "company_final", "reference_type",
              "sentiment", "relevance", "confidence", "ticker_guess", "ticker", "ticker_at_post", "entity_id", "match_method",
              "in_universe", "pair_reviewed", "review_reason", "llm_model", "llm_prompt_version"]
Twitter_tickers = refs.sort_values(["created_at", "ID", "ref_id"])[final_cols].reset_index(drop=True)

with sqlite3.connect(SOCIAL_DB) as conn:
    n_refs_db, n_posts_db = conn.execute("SELECT count(*), count(DISTINCT ID) FROM SocialMediaKeywords").fetchone()
assigning = {"ticker+name", "name", "executive", "manual"}
with_ticker = Twitter_tickers["ticker"].notna()
day = Twitter_tickers["created_at"].str[:10]
checks = {
    "every reference from SocialMediaKeywords appears": len(Twitter_tickers) == n_refs_db and Twitter_tickers["ID"].nunique() == n_posts_db,
    "no ticker without a review decision (pair_reviewed = 1)": (Twitter_tickers.loc[with_ticker, "pair_reviewed"] == 1).all(),
    "no ticker without an assigning match_method": Twitter_tickers.loc[with_ticker, "match_method"].isin(assigning).all(),
    "manual rows all have a ticker": Twitter_tickers.loc[Twitter_tickers["match_method"] == "manual", "ticker"].notna().all(),
    "assigning rules without a ticker only where the pair is pending": (Twitter_tickers.loc[Twitter_tickers["match_method"].isin(assigning) & ~with_ticker, "pair_reviewed"] == 0).all(),
    "every ticker exists in companies": Twitter_tickers.loc[with_ticker, "ticker"].isin(universe).all(),
    "every ticker has a price on its post date's range": not (with_ticker & ((day < Twitter_tickers["ticker"].map(series["first_price"])) | (day > Twitter_tickers["ticker"].map(series["last_price"])))).any(),
    "every ticker has an entity_id and a ticker_at_post": Twitter_tickers.loc[with_ticker, ["entity_id", "ticker_at_post"]].notna().all().all(),
    "every row has a match_method": Twitter_tickers["match_method"].notna().all(),
    "in_universe equals ticker set": (Twitter_tickers["in_universe"] == with_ticker.astype(int)).all(),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(checks.values())

method_order = ["ticker+name", "name", "executive", "manual", "none_reviewed", "none_unlisted", "ticker_only", "fuzzy", "ambiguous", "none", "indirect"]
method_counts = (Twitter_tickers.groupby("match_method").agg(rows=("ID", "size"), posts=("ID", "nunique"))
                 .reindex(method_order).fillna(0).astype(int))
method_counts["share_of_rows"] = (method_counts["rows"] / len(Twitter_tickers)).round(3)
display(method_counts)
moved = Twitter_tickers[with_ticker & (Twitter_tickers["ticker_at_post"] != Twitter_tickers["ticker"])]
print("References whose symbol on the post date differs from the latest symbol:", moved.groupby(["ticker", "ticker_at_post"]).size().to_dict())

Twitter_tickers.to_excel("Twitter_tickers_2025_2026.xlsx", sheet_name="Twitter_tickers", index=False)
print(f"{len(Twitter_tickers)} rows from {Twitter_tickers['ID'].nunique()} posts written to Twitter_tickers_2025_2026.xlsx; "
      f"{Twitter_tickers['in_universe'].sum()} rows with a universe ticker ({Twitter_tickers.loc[Twitter_tickers['in_universe'] == 1, 'ticker'].nunique()} distinct tickers)")

Symbol histories used for ticker_at_post: {'BNY': [('BK', '2026-05-21')], 'PPLI': [('IAC', '2026-06-04')], 'MSGK': [('MSGS', '2026-10-27')], 'SKYD': [('PARA', '2025-08-07'), ('PSKY', '2026-10-06')]}
OK  every reference from SocialMediaKeywords appears
OK  no ticker without a review decision (pair_reviewed = 1)
OK  no ticker without an assigning match_method
OK  manual rows all have a ticker
OK  assigning rules without a ticker only where the pair is pending
OK  every ticker exists in companies
OK  every ticker has a price on its post date's range
OK  every ticker has an entity_id and a ticker_at_post
OK  every row has a match_method
OK  in_universe equals ticker set


,rows,posts,share_of_rows
match_method,,,
ticker+name,1571,1354,0.243
name,7,7,0.001
executive,0,0,0.000
manual,104,100,0.016
none_reviewed,33,32,0.005
none_unlisted,8,8,0.001
ticker_only,0,0,0.000
fuzzy,0,0,0.000
ambiguous,0,0,0.000


References whose symbol on the post date differs from the latest symbol: {('MSGK', 'MSGS'): 3, ('PPLI', 'IAC'): 1, ('SKYD', 'PARA'): 22, ('SKYD', 'PSKY'): 59}


6468 rows from 4376 posts written to Twitter_tickers_2025_2026.xlsx; 1682 rows with a universe ticker (141 distinct tickers)


## 5) Descriptive analysis

`TwitterDescriptiveAnalysis.xlsx`, one sheet per table, for the data chapter: posts per month and account; the share of posts with at least one company reference; rows per `reference_type` and per `match_method`; the sentiment distribution and the positive / neutral / negative shares at the pre-registered thresholds (±0.5, with ±0.3 and ±0.7 as sensitivity); the 50 most-mentioned companies with ticker and whether they are in the trade universe; the share of distinct mentioned companies that are in the universe (the Part 1 number); posts blocked by the content filter per account and month; the pair decisions and the single-reference overrides of the review (section 3).

In [17]:
with sqlite3.connect(SOCIAL_DB) as conn:
    posts = pd.read_sql_query("SELECT ID, created_at, User, platform, llm_status FROM SocialMedia", conn)
posts["month"] = pd.to_datetime(posts["created_at"], utc=True, errors="coerce").dt.strftime("%Y-%m")
Twitter_tickers["month"] = pd.to_datetime(Twitter_tickers["created_at"], utc=True, errors="coerce").dt.strftime("%Y-%m")
company_rows = Twitter_tickers[Twitter_tickers["reference_type"] != "indirect"]

# 5.1 posts per month and account (all posts, labelled or blocked)
posts_per_month = posts.pivot_table(index="month", columns="User", values="ID", aggfunc="count", fill_value=0, margins=True, margins_name="total")

# 5.2 share of posts with at least one company reference (labelled posts only)
labelled = posts[posts["llm_status"] == "ok"]
with_ref = set(Twitter_tickers["ID"]); with_company = set(company_rows["ID"]); with_universe = set(company_rows.loc[company_rows["in_universe"] == 1, "ID"])
def share_table(group_col):
    g = labelled.groupby(group_col)["ID"]
    out = pd.DataFrame({"labelled_posts": g.size(),
                        "with_any_reference": g.apply(lambda s: s.isin(with_ref).sum()),
                        "with_company_reference": g.apply(lambda s: s.isin(with_company).sum()),
                        "with_universe_company": g.apply(lambda s: s.isin(with_universe).sum())})
    for c in ["with_any_reference", "with_company_reference", "with_universe_company"]:
        out["share_" + c] = (out[c] / out["labelled_posts"]).round(4)
    return out
reference_share = pd.concat([share_table("User"), share_table(pd.Series("all accounts", index=labelled.index, name="User"))])

# 5.3 rows per reference_type and per match_method
by_type = Twitter_tickers.groupby("reference_type").agg(rows=("ID", "size"), posts=("ID", "nunique"), in_universe_rows=("in_universe", "sum"))
by_type["share_in_universe"] = (by_type["in_universe_rows"] / by_type["rows"]).round(4)
by_method = method_counts.copy()
by_method_before = summary.rename(columns={"share_of_rows": "share_of_rows_before_review"})

# 5.4 sentiment distribution and classes on company rows and on in-universe rows
def sentiment_classes(df, label):
    rows = {}
    for thr in (0.5, 0.3, 0.7):
        rows[f"positive (>= {thr})"] = (df["sentiment"] >= thr).mean()
        rows[f"negative (<= -{thr})"] = (df["sentiment"] <= -thr).mean()
        rows[f"neutral (between)"] = 1 - rows[f"positive (>= {thr})"] - rows[f"negative (<= -{thr})"]
        rows[f"neutral (|s| < {thr})"] = rows.pop("neutral (between)")
    return pd.Series(rows, name=label).round(4)
sentiment_shares = pd.concat([sentiment_classes(company_rows, "company rows"), sentiment_classes(company_rows[company_rows["in_universe"] == 1], "in-universe rows"),
                              sentiment_classes(Twitter_tickers, "all rows incl. indirect")], axis=1)
sentiment_distribution = pd.concat([company_rows["sentiment"].value_counts().sort_index().rename("company rows"),
                                    company_rows.loc[company_rows["in_universe"] == 1, "sentiment"].value_counts().sort_index().rename("in-universe rows")], axis=1).fillna(0).astype(int)

# 5.5 the 50 most-mentioned companies: universe ticker where established, else the LLM's name
company_rows = company_rows.assign(company_key=company_rows["ticker"].where(company_rows["ticker"].notna(), company_rows["canonical_company"].apply(standardize_company)))
top_companies = (company_rows.groupby("company_key").agg(rows=("ID", "size"), posts=("ID", "nunique"), ticker=("ticker", "first"),
                                                          in_universe=("in_universe", "max"), example_name=("canonical_company", "first"),
                                                          mean_sentiment=("sentiment", "mean"))
                 .sort_values("rows", ascending=False).head(50).round({"mean_sentiment": 2}))
top_companies["company_name"] = top_companies["ticker"].map(ticker_to_fmp).fillna(top_companies["example_name"])
top_companies = top_companies[["company_name", "ticker", "in_universe", "rows", "posts", "mean_sentiment"]]

# 5.6 Part 1: share of distinct mentioned companies that are in the universe
distinct_companies = company_rows["company_key"].nunique()
distinct_in_universe = company_rows.loc[company_rows["in_universe"] == 1, "ticker"].nunique()
part1 = pd.DataFrame({
    "measure": ["distinct companies mentioned (company rows)", "of which in the trade universe", "share of distinct companies in universe (Part 1)",
                "company references (rows)", "of which with a universe ticker", "share of company references in universe",
                "posts with a company reference", "of which with a universe company", "share of posts in universe"],
    "value": [distinct_companies, distinct_in_universe, round(distinct_in_universe / distinct_companies, 4),
              len(company_rows), int(company_rows["in_universe"].sum()), round(company_rows["in_universe"].mean(), 4),
              len(with_company), len(with_universe), round(len(with_universe) / len(with_company), 4)]})

# 5.7 posts blocked by the content filter per account and month
blocked = posts[posts["llm_status"] == "content_filter"].pivot_table(index="month", columns="User", values="ID", aggfunc="count", fill_value=0, margins=True, margins_name="total")

# 5.8 the review decisions: one row per pair, and the single-reference overrides
reviewed_pairs = pair_decisions[PAIR_COLS]
reviewed_rows = row_overrides[ROW_COLS]

with pd.ExcelWriter("TwitterDescriptiveAnalysis.xlsx", engine="openpyxl") as writer:
    posts_per_month.to_excel(writer, sheet_name="PostsPerMonth")
    reference_share.to_excel(writer, sheet_name="ReferenceShare")
    by_type.to_excel(writer, sheet_name="ByReferenceType")
    by_method.to_excel(writer, sheet_name="ByMatchMethod")
    by_method_before.to_excel(writer, sheet_name="ByMatchMethodBeforeReview")
    sentiment_shares.to_excel(writer, sheet_name="SentimentShares")
    sentiment_distribution.to_excel(writer, sheet_name="SentimentDistribution")
    top_companies.to_excel(writer, sheet_name="Top50Companies")
    part1.to_excel(writer, sheet_name="Part1Share", index=False)
    blocked.to_excel(writer, sheet_name="BlockedPosts")
    reviewed_pairs.to_excel(writer, sheet_name="ReviewedPairs", index=False)
    reviewed_rows.to_excel(writer, sheet_name="ReviewedRows", index=False)
print("TwitterDescriptiveAnalysis.xlsx written: 12 sheets")
display(part1)
display(reference_share)
display(by_type)
display(sentiment_shares)
display(top_companies.head(20))

TwitterDescriptiveAnalysis.xlsx written: 12 sheets


,measure,value
0,distinct companies mentioned (company rows),567.0000
1,of which in the trade universe,141.0000
2,share of distinct companies in universe (Part 1),0.2487
3,company references (rows),2819.0000
4,of which with a universe ticker,1682.0000
5,share of company references in universe,0.5967
6,posts with a company reference,2248.0000
7,of which with a universe company,1432.0000
8,share of posts in universe,0.6370


,labelled_posts,with_any_reference,with_company_reference,with_universe_company,share_with_any_reference,share_with_company_reference,share_with_universe_company
User,,,,,,,
POTUS,105,18,11,6,0.1714,0.1048,0.0571
WhiteHouse,16060,3113,1523,955,0.1938,0.0948,0.0595
realDonaldTrump,5777,1245,714,471,0.2155,0.1236,0.0815
all accounts,21942,4376,2248,1432,0.1994,0.1025,0.0653


,rows,posts,in_universe_rows,share_in_universe
reference_type,,,,
brand,412,372,250,0.6068
direct,2246,1803,1296,0.5770
executive,161,153,136,0.8447
indirect,3649,2395,0,0.0000


,company rows,in-universe rows,all rows incl. indirect
positive (>= 0.5),0.3366,0.3252,0.3695
negative (<= -0.5),0.1767,0.1801,0.1647
neutral (|s| < 0.5),0.4867,0.4946,0.4658
positive (>= 0.3),0.3413,0.3323,0.3724
negative (<= -0.3),0.1774,0.1807,0.1659
neutral (|s| < 0.3),0.4814,0.4869,0.4617
positive (>= 0.7),0.1181,0.1112,0.0829
negative (<= -0.7),0.1554,0.1617,0.1127
neutral (|s| < 0.7),0.7265,0.7271,0.8044


,company_name,ticker,in_universe,rows,posts,mean_sentiment
company_key,,,,,,
FOXA,Fox Corporation,FOXA,1,336,335,0.10
NWSA,News Corporation,NWSA,1,121,118,-0.29
TSLA,"Tesla, Inc.",TSLA,1,121,121,0.28
THE NEW YORK TIMES COMPANY,The New York Times Company,None,0,107,107,-0.82
WBD,"Warner Bros. Discovery, Inc.",WBD,1,93,93,-0.71
CMCSA,Comcast Corporation,CMCSA,1,85,85,-0.44
SKYD,Skydance Corporation,SKYD,1,81,80,-0.31
DIS,The Walt Disney Company,DIS,1,80,78,-0.46
TKO,"TKO Group Holdings, Inc.",TKO,1,75,75,0.26


→ We now have one row per post and company with a validated ticker from the trade universe, or an explicit "not in universe", ready for a pandas join on `ticker`.